# Enterprise clients

From [Forward Deployed Engineering](https://gradlabs.in/learn/fde/lessons/enterprise.html). The client, Northstar Skills, is made up: every name, number and message is invented.

Your company's agent now runs for two clients: Northstar Skills and Meridian Learning, a bigger chain with its own IT team. Both clients' leads sit in one store.

Make it safe for both: find a data leak and fix it, decide who may do what, check sign-in tokens, handle webhooks that can be faked or sent twice, and keep a log Kevin can audit.

No API key needed. For each cell: **guess what it will print first**, then press ▶ and check.

**1. Get the code.**

In [ ]:
import os
if not os.path.exists("/content/forward-deployed-engineering"):
    !git clone -q --depth 1 https://github.com/mo-ri-ch/forward-deployed-engineering /content/forward-deployed-engineering
%cd /content/forward-deployed-engineering/starter/enterprise

**2. Look at the store.** One table, two clients. Each counsellor works at one centre.

In [ ]:
import pandas as pd

leads = pd.read_csv("data/leads.csv")
print(leads.groupby(["client", "centre"]).size())
leads.head(10)

**3. Make the leak happen.** Priya is a Meridian counsellor at Centre 4. She searches for "arun".

🤔 *How many results should she see? How many will she?*

In [ ]:
from store import LeadStore
from users import USERS

store = LeadStore()
for r in store.search(USERS["priya@meridian"], "arun"):
    print(r["client"], r["lead_id"], r["name"], r["phone"])

She sees Northstar's Arun, phone number and all. That one line is the whole risk: a client seeing another client's data ends the contract, for both.

**4. Test for leaks before fixing them.** This check signs in as every user and searches for every name. Write the test first, so you know when the fix works.

In [ ]:
from checks import check_isolation

check_isolation(store.search)

**5. Fix it.** Every query filters by the signed-in user's client. Counsellors also only see their own centre.

The client comes from the sign-in, never from the request, so nobody can ask for another client's data by changing a URL.

In [ ]:
def search(user, text):
    rows = [r for r in store.rows if r["client"] == user["client"]]          # this client only
    if user["role"] == "counsellor":
        rows = [r for r in rows if r["centre"] == user["centre"]]        # and her own centre
    return [r for r in rows if text.lower() in r["name"].lower()]

check_isolation(search)

🧪 *Try it:* delete the client line and run again. The check should catch it.

**6. Who may do what.** Write the rules as a short table, then one function that only says yes when a rule allows it. Anything not on the list is a no.

In [ ]:
RULES = {
    "counsellor": {"view"},
    "manager": {"view", "export"},
}

def can(user, action, lead):
    if lead["client"] != user["client"]:
        return False
    if action not in RULES.get(user["role"], set()):
        return False
    if user["role"] == "counsellor" and lead["centre"] != user["centre"]:
        return False
    return True

from checks import check_can
check_can(can)

**7. Check who's signing in.** With SSO, Meridian's login system sends your app a signed note: who this is and until when. Your app checks the signature with the shared key before trusting the note.

This is a simplified version. Real SSO uses public keys and a library such as `authlib`, but the checks are the same.

In [ ]:
import hashlib, hmac, time
from tokens import make_token, read_token, MERIDIAN_LOGIN_KEY, _b64

def sign_in(token):
    body, sig, claims = read_token(token)
    good = _b64(hmac.new(MERIDIAN_LOGIN_KEY, body.encode(), hashlib.sha256).digest())
    if not hmac.compare_digest(sig, good):
        return "rejected: signature doesn't match"
    if claims["expires"] < time.time():
        return "rejected: expired"
    if claims["email"] not in USERS:
        return "rejected: not a user here"
    return "signed in as " + claims["email"]

now = time.time()
real = make_token({"email": "priya@meridian", "expires": now + 3600})
old = make_token({"email": "priya@meridian", "expires": now - 60})
fake = make_token({"email": "sanjay@meridian", "expires": now + 3600}, key=b"a-guessed-key")

for name, t in [("real", real), ("expired", old), ("forged", fake)]:
    print(f"{name:8}", sign_in(t))

🤔 *Divya leaves Northstar. IT disables her account. Why does her access end everywhere at once?* Her next sign-in fails at Northstar's login system, and her token expires within the hour. No one has to remember your app.

**8. Webhooks.** Meridian's CRM calls your web address when a lead changes. Two things go wrong in real life: someone sends a fake call, and the CRM sends the same event twice when it retries.

🤔 *There are 6 calls below. How many should change a lead?*

In [ ]:
import json
import webhooks

calls = webhooks.load()
for c in calls:
    e = json.loads(c["body"])
    print(e["event_id"], e["type"], e["lead_id"])

In [ ]:
seen = set()
status = {}

def handle(call):
    good = "sha256=" + hmac.new(webhooks.WEBHOOK_SECRET, call["body"].encode(), hashlib.sha256).hexdigest()
    if not hmac.compare_digest(call["headers"]["X-Meridian-Signature"], good):
        return 401, "rejected: bad signature"
    event = json.loads(call["body"])
    if event["event_id"] in seen:
        return 200, "already done"       # say OK, so the CRM stops retrying
    seen.add(event["event_id"])
    status[event["lead_id"]] = event["type"]
    return 200, "updated " + event["lead_id"]

for c in calls:
    print(*handle(c))
print()
print(status)

Four leads changed. The fake "enrolled" for M-204 was refused, and the repeat of ev_9002 did nothing. Handling the same event twice safely is called being **idempotent**.

🧪 *Try it:* run the loop again. Does anything change?

**9. Keep an audit log.** Kevin's review asks: *can you show who looked at a lead?* Write a line every time someone views or exports, and never edit old lines.

In [ ]:
from datetime import datetime
audit = []

def view(uid, lead_id):
    user = USERS[uid]
    lead = next(r for r in store.rows if r["lead_id"] == lead_id)
    allowed = can(user, "view", lead)
    audit.append({"time": datetime.now().strftime("%H:%M:%S"), "who": uid, "action": "view",
                  "lead": lead_id, "allowed": allowed})
    return lead if allowed else None

view("divya@northstar", "N-102")
view("anand@northstar", "N-102")   # another centre: refused, and logged
view("kevin@northstar", "N-102")
view("sanjay@meridian", "N-102")   # another client: refused, and logged

pd.DataFrame([a for a in audit if a["lead"] == "N-102"])

**10. Answer Kevin's review with evidence.** Each answer points to something you ran, not a promise.

In [ ]:
print("Can another client see our leads?   No.", "Leak test:", "passed" if check_isolation(search) else "FAILED")
print("Can counsellors export?              No. Rules:", RULES)
print("Are fake webhooks refused?           Yes.", sum(handle(c)[0] == 401 for c in calls), "of", len(calls), "refused in the test")
print("Who looked at lead N-102?           ", ", ".join(a["who"] for a in audit if a["lead"] == "N-102"))

**What you've built:** data that stays with its client, rules that say no by default, checked sign-ins, safe webhooks and an audit log. These are the questions every large client asks before signing.

Back to the course: [Enterprise clients](https://gradlabs.in/learn/fde/lessons/enterprise.html).